# Week 03 — Python Solution Lab
## Projectile Motion (2D Kinematics)

**Companion to `notebooks/Week_03.ipynb`.** This notebook contains *fully worked Python
solutions* to selected problems from that week's problem set — one at **each difficulty level**.

Every solution follows the course's core workflow:

> **Diagram → Principle → Equation → Predict → Verify**

The markdown cell states the problem, identifies the governing principle, and gives the **hand
prediction you should make before running anything**. The code cell then computes the result and
*verifies* it — typically by a second independent method (energy vs. forces, symbolic vs.
numerical, closed form vs. simulation) — and includes `assert` checks against the known answer.

### How to use this notebook

1. **Attempt the problem in `Week_03.ipynb` first.** These solutions are worth very little
   if you read them before trying.
2. Make the hand prediction. Write it down.
3. Run the code cell and compare.
4. **Change a number and re-run.** Every solution is written so that the parameters sit at the
   top; the sweeps and plots update automatically. Ask "what if the mass doubled?" and answer it
   in ten seconds.

### Why the code looks like this

These are not minimal answer-generators. Each one demonstrates something Python does that hand
algebra cannot: parameter sweeps, root-finding, numerical integration, symbolic differentiation,
or a cross-check to machine precision. The physics is the point; the code is how we prove the
physics is right.

---


### Solutions in this notebook

| Level | Problem | Topic | Python technique |
|---|---|---|---|
| **L1 · Basic** | `P2` | Horizontal Launch | strobe plot proving axis independence |
| **L2 · Intermediate** | `P6` | Complementary Angles | intermediate-point constraint + sensitivity |
| **L3 · Challenge** | `P9` | Projectile on an Incline | `np.roots`, `brentq`, angle optimisation |

---

## L1 · Basic — P2: Horizontal Launch

> **Problem (Week_03.ipynb, L1 — P2).** A marble rolls off the edge of a table $1.20$ m above
> the floor with a horizontal speed of $2.50$ m/s. (a) How long to hit the floor?
> (b) How far from the base of the table does it land?

**Diagram → Principle.** The two axes are **independent**: gravity acts only on $y$, so the fall
time is exactly the free-fall time from $1.20$ m, regardless of how fast the marble was rolling.

**Equation.** $t = \sqrt{2h/g}$, then $x = v_0 t$.

**Hand prediction.** $t = 0.495$ s, $x = 1.24$ m.

**What Python adds.** We *prove* axis independence by dropping a second marble straight down in
the same simulation and showing the two have identical $y(t)$ at every instant — the single most
counter-intuitive idea in the week, made undeniable in three lines.

In [ ]:
# ═══ W03 · L1 · P2 — Horizontal launch, and a proof that the axes are independent ═══
import numpy as np
import matplotlib.pyplot as plt

# --- MODEL --------------------------------------------------------------
h, v0, g = 1.20, 2.50, 9.81      # m, m/s, m/s^2

# --- PREDICT ------------------------------------------------------------
t_fall = np.sqrt(2 * h / g)
x_land = v0 * t_fall
print(f"(a) t = sqrt(2h/g) = {t_fall:.3f} s   <- note: v0 does NOT appear")
print(f"(b) x = v0 * t     = {x_land:.3f} m")

# --- VERIFY: simulate BOTH the rolled marble and a dropped one ----------
t   = np.linspace(0, t_fall, 400)
x_r,  y_r  = v0 * t,            h - 0.5 * g * t**2     # rolled off the edge
x_d,  y_d  = np.zeros_like(t),  h - 0.5 * g * t**2     # simply dropped

print(f"\nmax |y_rolled - y_dropped| over the whole fall = "
      f"{np.max(np.abs(y_r - y_d)):.1e} m")
print("  -> the horizontal motion has ZERO effect on the vertical motion.")
assert np.allclose(y_r, y_d)

# --- Impact conditions --------------------------------------------------
vx, vy = v0, -g * t_fall
print(f"\nImpact velocity: vx = {vx:.2f} m/s (unchanged), vy = {vy:.2f} m/s")
print(f"  speed = {np.hypot(vx, vy):.2f} m/s at {np.degrees(np.arctan2(vy, vx)):.1f} deg "
      "below horizontal")

# --- Plot: strobe view --------------------------------------------------
fig, ax = plt.subplots(figsize=(6.6, 4))
ax.plot(x_r, y_r, color="#1565c0", lw=2, label="rolled off at 2.50 m/s")
ax.plot(x_d, y_d, color="#e65100", lw=2, ls="--", label="dropped from rest")
strobe = np.linspace(0, t_fall, 9)
ax.plot(v0*strobe, h - .5*g*strobe**2, "o", color="#1565c0", ms=6)
ax.plot(0*strobe,  h - .5*g*strobe**2, "s", color="#e65100", ms=6)
for s in strobe:                                   # equal-time tie lines
    ax.plot([0, v0*s], [h - .5*g*s**2]*2, color="grey", lw=.7, ls=":")
ax.axhline(0, c="k", lw=1.2)
ax.set_xlabel("x (m)"); ax.set_ylabel("y (m)")
ax.set_title("W03 P2 — equal-time strobe: both marbles fall together")
ax.grid(alpha=.3); ax.legend()
plt.tight_layout(); plt.show()

# --- CHECK --------------------------------------------------------------
assert abs(t_fall - 0.495) < 0.002 and abs(x_land - 1.24) < 0.01
print("[OK] Matches textbook answer: t = 0.495 s, x = 1.24 m")

## L2 · Intermediate — P6: Complementary Angles

> **Problem (Week_03.ipynb, L2 — P6).** A cannon fires at $50.0$ m/s. (a) Find the two launch
> angles that give a range of $200$ m on level ground. (b) For each, find max height and flight
> time. (c) Which trajectory clears a $15$ m wall $150$ m away?

**Diagram → Principle.** $R = \frac{v_0^2\sin 2\theta}{g}$, and $\sin2\theta = \sin(180^\circ-2\theta)$,
so $\theta$ and $90^\circ-\theta$ share a range.

**Equation.** $\theta = \tfrac12\arcsin\!\big(Rg/v_0^2\big)$ and its complement.

**Hand prediction.** $\sin2\theta = 200(9.81)/2500 = 0.7848 \Rightarrow \theta = 25.8^\circ$ or $64.2^\circ$.

**What Python adds — and a warning about guessing.** Part (c) is a *constraint check at an
intermediate point*: evaluate $y(x=150)$ for both trajectories. The intuitive answer is "the flat
one is too low" — **but run the numbers and it is not**: the flat shot passes $18.2$ m and clears
the $15$ m wall. (The printed answer key gets this right, and picks the flat shot for its shorter
flight time — a perfectly good reason.) What the numbers add is the **margin**: $3.2$ m versus
$62.4$ m. Below we perturb the muzzle speed by a few percent and watch the flat trajectory lose
its margin entirely while the lofted one never comes close. Shorter flight time or more
robustness — that is a real engineering trade-off, and you can only see it once you compute it.

In [ ]:
# ═══ W03 · L2 · P6 — Complementary launch angles and a wall-clearance test ═══
import numpy as np
import matplotlib.pyplot as plt

# --- MODEL --------------------------------------------------------------
v0, g, R_target = 50.0, 9.81, 200.0
wall_x, wall_h  = 150.0, 15.0

# --- (a) PREDICT: solve R = v0^2 sin(2 theta)/g -------------------------
s = R_target * g / v0**2
assert s <= 1, "target range is beyond the maximum possible range"
th1 = 0.5 * np.arcsin(s)                 # the shallow root
th2 = np.pi / 2 - th1                     # the complementary, lofted root
print(f"(a) sin(2*theta) = {s:.4f}")
print(f"    theta_1 = {np.degrees(th1):.2f} deg   (flat)")
print(f"    theta_2 = {np.degrees(th2):.2f} deg   (lofted)   -- they sum to 90 deg")

# --- (b) height and time for each ---------------------------------------
def flight(theta):
    T = 2 * v0 * np.sin(theta) / g
    H = (v0 * np.sin(theta))**2 / (2 * g)
    Rg = v0**2 * np.sin(2 * theta) / g
    return T, H, Rg

print("\n(b)   theta (deg)   T (s)    H (m)    R (m)")
for th in (th1, th2):
    T, H, Rg = flight(th)
    print(f"      {np.degrees(th):9.2f}   {T:6.2f}   {H:6.2f}   {Rg:6.1f}")
    assert abs(Rg - R_target) < 1e-6, "both angles must give the same range"

# --- (c) clearance test at x = 150 m ------------------------------------
def y_at_x(theta, x):
    """Trajectory height at horizontal distance x (eliminating t)."""
    return x * np.tan(theta) - g * x**2 / (2 * (v0 * np.cos(theta))**2)

print(f"\n(c) height above the ground at x = {wall_x:.0f} m (wall is {wall_h:.0f} m tall):")
for name, th in (("flat  ", th1), ("lofted", th2)):
    y = y_at_x(th, wall_x)
    verdict = "CLEARS" if y > wall_h else "HITS THE WALL"
    print(f"    {name} ({np.degrees(th):5.2f} deg): y = {y:6.2f} m  ->  {verdict}"
          f"   (margin {y - wall_h:+6.2f} m)")
print("    Both clear -- the intuitive 'the flat one is too low' guess is WRONG.")

# So the deciding factor is ROBUSTNESS, not a yes/no. Perturb the muzzle speed.
print("\n    Sensitivity: same launch angles, muzzle speed off by a few percent")
print("      dv0     flat y(150)     lofted y(150)")
for pct in (-8, -6, -4, -2, 0, +2, +4):
    v  = 50.0 * (1 + pct / 100)
    yf = wall_x*np.tan(th1) - g*wall_x**2 / (2*(v*np.cos(th1))**2)
    yl = wall_x*np.tan(th2) - g*wall_x**2 / (2*(v*np.cos(th2))**2)
    flag = "   <- FLAT SHOT NOW HITS THE WALL" if yf < wall_h else ""
    print(f"      {pct:+3d}%   {yf:9.2f} m     {yl:9.2f} m{flag}")
print("    -> the lofted shot is the robust choice: it keeps ~60 m of clearance, while")
print("       the flat one burns its 3.2 m margin after only a few percent of speed error.")

# --- Plot ---------------------------------------------------------------
xs = np.linspace(0, R_target, 500)
fig, ax = plt.subplots(figsize=(8, 4))
for th, col, lbl in ((th1, "#1565c0", "flat"), (th2, "#2e7d32", "lofted")):
    ax.plot(xs, y_at_x(th, xs), color=col, lw=2,
            label=f"{lbl}: {np.degrees(th):.1f} deg")
ax.plot([wall_x, wall_x], [0, wall_h], color="crimson", lw=6, solid_capstyle="butt",
        label=f"{wall_h:.0f} m wall at {wall_x:.0f} m")
ax.axhline(0, c="k", lw=1)
ax.set_xlabel("x (m)"); ax.set_ylabel("y (m)"); ax.set_ylim(bottom=0)
ax.set_title("W03 P6 — same range, very different paths")
ax.grid(alpha=.3); ax.legend()
plt.tight_layout(); plt.show()

# --- CHECK --------------------------------------------------------------
assert abs(np.degrees(th1) - 25.8) < 0.1 and abs(np.degrees(th2) - 64.2) < 0.1
assert y_at_x(th2, wall_x) > y_at_x(th1, wall_x) > wall_h, "at nominal speed both clear"
assert abs(y_at_x(th1, wall_x) - 18.17) < 0.05 and abs(y_at_x(th2, wall_x) - 77.40) < 0.05
print("[OK] Matches textbook answer: 25.8 deg and 64.2 deg give the same 200 m range.")
print("     At the nominal speed BOTH clear the wall -- but only the lofted shot does so robustly.")

## L3 · Challenge — P9: Projectile on an Incline

> **Problem (Week_03.ipynb, L3 — P9).** A ball is launched from the base of a hill of slope
> $\alpha = 20^\circ$ at $25.0$ m/s and $50^\circ$ above the **horizontal**.
> (a) Find the impact time (hint: impact when $y/x = \tan\alpha$). (b) Find the distance along
> the incline to the impact point.

**Diagram → Principle.** The ground is no longer $y = 0$ but the line $y = x\tan\alpha$. The
landing condition becomes an equation in $t$.

**Equation.** $v_0\sin\theta\,t - \tfrac12 gt^2 = (v_0\cos\theta\,t)\tan\alpha$, which factors to
$t\big[v_0(\sin\theta - \cos\theta\tan\alpha) - \tfrac12 gt\big] = 0$.

**Hand prediction.** $t = \dfrac{2v_0(\sin\theta - \cos\theta\tan\alpha)}{g}$.

**What Python adds.** Two things. First, we solve the landing condition **three ways** —
factored algebra, `np.roots`, and `scipy.optimize.brentq` on the gap function — and show they
agree; that is the toolkit you need when the surface *isn't* a straight line. Second, we sweep
the launch angle to find the one that maximises up-slope range, recovering the classic
$\theta_{\text{best}} = \alpha/2 + 45^\circ$ result numerically.

In [ ]:
# ═══ W03 · L3 · P9 — Landing on a slope: three solution routes + an optimisation ═══
import numpy as np
from scipy.optimize import brentq
import matplotlib.pyplot as plt

# --- MODEL --------------------------------------------------------------
v0, g       = 25.0, 9.81
theta       = np.radians(50.0)      # launch angle above the HORIZONTAL
alpha       = np.radians(20.0)      # slope of the hill
vx, vy      = v0 * np.cos(theta), v0 * np.sin(theta)

x  = lambda t: vx * t
y  = lambda t: vy * t - 0.5 * g * t**2
gap = lambda t: y(t) - x(t) * np.tan(alpha)      # vertical gap above the slope

# --- (a) ROUTE 1: factored algebra --------------------------------------
t_alg = 2 * v0 * (np.sin(theta) - np.cos(theta) * np.tan(alpha)) / g
# --- ROUTE 2: polynomial roots of  -g/2 t^2 + (vy - vx tan a) t = 0 ------
t_poly = max(np.roots([-0.5 * g, vy - vx * np.tan(alpha), 0.0]))
# --- ROUTE 3: numerical root of the gap function ------------------------
t_num = brentq(gap, 1e-6, 2 * vy / g)

print(f"(a) impact time")
print(f"    algebra          t = {t_alg:.4f} s")
print(f"    np.roots         t = {t_poly:.4f} s")
print(f"    brentq(gap)      t = {t_num:.4f} s")
assert np.allclose([t_alg, t_poly, t_num], t_alg, atol=1e-6)

# --- (b) distance measured ALONG the incline ----------------------------
xi, yi = x(t_alg), y(t_alg)
R_slope = np.hypot(xi, yi)
print(f"\n(b) impact point (x, y) = ({xi:.3f}, {yi:.3f}) m")
print(f"    range along the incline R = {R_slope:.3f} m")
print(f"    check: y/x = {yi/xi:.4f} vs tan(alpha) = {np.tan(alpha):.4f}")
assert abs(yi / xi - np.tan(alpha)) < 1e-9

# --- OPTIMISE: which launch angle goes furthest up the slope? -----------
def slope_range(th):
    vx_, vy_ = v0*np.cos(th), v0*np.sin(th)
    tt = 2 * v0 * (np.sin(th) - np.cos(th)*np.tan(alpha)) / g
    return np.hypot(vx_*tt, vy_*tt - 0.5*g*tt**2) if tt > 0 else 0.0

ths   = np.radians(np.linspace(alpha + 0.5, 89.5, 2000))
Rs    = np.array([slope_range(t_) for t_ in ths])
best  = ths[np.argmax(Rs)]
print(f"\nOptimum launch angle (swept): {np.degrees(best):.2f} deg, R = {Rs.max():.2f} m")
print(f"Classic result alpha/2 + 45 deg = {np.degrees(alpha)/2 + 45:.2f} deg  -> agrees")
assert abs(np.degrees(best) - (np.degrees(alpha)/2 + 45)) < 0.2

# --- Plot ---------------------------------------------------------------
tt = np.linspace(0, t_alg, 300)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.5, 4))
ax1.plot(x(tt), y(tt), color="#1565c0", lw=2, label="trajectory (50 deg)")
xs = np.linspace(0, x(t_alg) * 1.15, 50)
ax1.plot(xs, xs * np.tan(alpha), color="#5d4037", lw=2, label="hill (20 deg)")
ax1.fill_between(xs, 0, xs*np.tan(alpha), color="#5d4037", alpha=.15)
ax1.plot(xi, yi, "o", color="crimson", ms=9, zorder=5, label="impact")
ax1.set_xlabel("x (m)"); ax1.set_ylabel("y (m)"); ax1.set_aspect("equal")
ax1.grid(alpha=.3); ax1.legend(); ax1.set_title("trajectory vs incline")

ax2.plot(np.degrees(ths), Rs, color="#2e7d32", lw=2)
ax2.axvline(np.degrees(best), ls="--", c="crimson", label=f"best {np.degrees(best):.1f} deg")
ax2.axvline(50, ls=":", c="grey", label="this problem: 50 deg")
ax2.set_xlabel("launch angle (deg)"); ax2.set_ylabel("range along slope (m)")
ax2.grid(alpha=.3); ax2.legend(); ax2.set_title("up-slope range vs angle")
plt.suptitle("W03 P9 — projectile onto a 20 deg incline", y=1.02)
plt.tight_layout(); plt.show()

# --- CHECK --------------------------------------------------------------
assert abs(t_alg - 2.7120) < 5e-3, "impact time"
assert abs(R_slope - 46.38) < 0.3, "slope range"
print(f"[OK] t = {t_alg:.3f} s, R along incline = {R_slope:.2f} m, "
      "confirmed by three independent methods.")

---

## Self-check

**Every code cell above contains one or more `assert` checks.** If you run the whole notebook top
to bottom without an `AssertionError`, all of the numerical checks on this page have passed.
(The asserts sit just before each cell's closing summary, so the last thing you see is a printed
result — not the check itself.)

**Now transfer the skill.** Pick one unsolved problem from `Week_03.ipynb` at the level you
found hardest, and write the same five-part structure for it:

```python
# --- MODEL:   parameters at the top, with units in comments
# --- PREDICT: the closed-form answer
# --- VERIFY:  a SECOND, independent route to the same number
# --- CHECK:   assert against your hand prediction
```

The verify step is the one that matters. A result you have only computed one way is a result you
have not checked.
